# GeoBrix extras install probe

**Purpose:** Validate that the `geobrix[light,stac,vizx,overture]` extras install cleanly
and that sensitive transitive deps resolve to compatible versions.

**Serverless / JOB usage:** This notebook has no `%pip` cells. Dependencies are
injected via the Serverless environment spec by the runner
(`gbx:test:notebooks-serverless --extras light,stac,vizx,overture`).

**Classic / `%pip` usage:** Wrap this notebook with a `%pip install` + `restartPython()`
preamble in a separate job notebook (see `task-4-brief.md` Step 4).

Exits with `dbutils.notebook.exit(json)` — result readable via
`jobs.get_run_output(task_run_id).notebook_output.result`.

In [ ]:
import importlib.metadata
import json

# ── Light surface: core three packages ────────────────────────────────────
import_errors = {}

try:
    from databricks.labs.gbx import pyrx  # noqa: F401
    print('pyrx: OK')
except Exception as exc:
    import_errors['pyrx'] = str(exc)
    print(f'pyrx: FAIL — {exc}')

try:
    from databricks.labs.gbx import pyvx  # noqa: F401
    print('pyvx: OK')
except Exception as exc:
    import_errors['pyvx'] = str(exc)
    print(f'pyvx: FAIL — {exc}')

try:
    from databricks.labs.gbx import pygx  # noqa: F401
    print('pygx: OK')
except Exception as exc:
    import_errors['pygx'] = str(exc)
    print(f'pygx: FAIL — {exc}')

# ── Feature module surfaces (stac / vizx / overture) ──────────────────────
try:
    from databricks.labs.gbx import stac  # noqa: F401
    print('stac: OK')
except Exception as exc:
    import_errors['stac'] = str(exc)
    print(f'stac: FAIL — {exc}')

try:
    from databricks.labs.gbx import vizx  # noqa: F401
    print('vizx: OK')
except Exception as exc:
    import_errors['vizx'] = str(exc)
    print(f'vizx: FAIL — {exc}')

try:
    from databricks.labs.gbx.sample import overture  # noqa: F401
    print('overture: OK')
except Exception as exc:
    import_errors['overture'] = str(exc)
    print(f'overture: FAIL — {exc}')

In [ ]:
# ── Sensitive transitive dependency versions ───────────────────────────────
def _ver(pkg: str) -> str:
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return 'NOT_INSTALLED'

versions = {
    'protobuf':           _ver('protobuf'),
    'mapbox_vector_tile': _ver('mapbox-vector-tile'),
    'idna':               _ver('idna'),
    'botocore':           _ver('botocore'),
    'rio_tiler':          _ver('rio-tiler'),
}

imports_ok = len(import_errors) == 0
all_ok = imports_ok

summary = {
    'env':           'serverless-v6',
    'extra':         'light,stac,vizx,overture',
    'imports_ok':    imports_ok,
    'import_errors': import_errors,
    'versions':      versions,
    'all_ok':        all_ok,
}

print(json.dumps(summary, indent=2))
dbutils.notebook.exit(json.dumps(summary))